## Basic Imports for Testing

In [1]:
import os
import finnhub
import pandas as pd
import json
import duckdb
from functools import reduce
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
api_key = os.getenv('FINNHUB_API_KEY')

# Setup client
finnhub_client = finnhub.Client(api_key=api_key)
test_symbol = 'AAPL'
test_symbol_name = 'apple'
test_symbol_list = ['AAPL', 'TSLA']
test_symbol_list_names = ['apple', 'tesla']

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

def normalize_json_dataframe(df):
    """
    Iterates through each row in the DataFrame, normalizes JSON content in each column,
    and creates new DataFrames from the normalized JSON, associating each resulting row
    with the original index value.
    
    Assumes each cell contains a JSON string, dict, or list of dicts.
    Returns a list of DataFrames, one per original row.
    """
    result_dataframes = {}
    
    for idx, row in df.iterrows():
        row_normalized_dfs = []
        
        for col in df.columns:
            cell_value = row[col]
            
            # Parse JSON if it's a string
            if isinstance(cell_value, str):
                try:
                    json_data = json.loads(cell_value)
                except json.JSONDecodeError:
                    continue  # Skip invalid JSON
            else:
                json_data = cell_value
            
            # Normalize if it's a dict or list of dicts
            if isinstance(json_data, (dict, list)):
                try:
                    normalized_df = pd.json_normalize(json_data)
                    normalized_df['original_index'] = idx  # Add original index to the normalized DataFrame
                    rename_cols = [i for i in normalized_df.columns if i == 'v']
                    for i in rename_cols:
                        normalized_df.rename(columns={i: f"{col}_{i}"}, inplace=True)
                    row_normalized_dfs.append(normalized_df)
                except Exception:
                    continue  # Skip if normalization fails
        
        # Combine all normalized DataFrames for this row
        if row_normalized_dfs:
            combined_df = reduce(lambda left, right: pd.merge(left, right, on=['period', 'original_index'], how='outer'), row_normalized_dfs)
            ## combined_df.set_index('original_index', inplace=True)
            result_dataframes[idx] = combined_df
    
    return result_dataframes

## Free Plan Endpoints

##### Company Basic Financials

In [48]:
# Basic financials
## print(finnhub_client.company_basic_financials(test_symbol, 'all'))
df = pd.DataFrame(finnhub_client.company_basic_financials(test_symbol, 'all'))
annual_and_quarterly_pre_df = df.copy(deep=True)

annual_and_quarterly_pre_df.index.name = 'index'
annual_and_quarterly_pre_df.reset_index(inplace=True)

annual_pre_df = pd.DataFrame(annual_and_quarterly_pre_df[annual_and_quarterly_pre_df['index'] == 'annual']).iloc[-2:,:].set_index('symbol') ## .reset_index()
## annual_pre_df = pd.DataFrame(annual_and_quarterly_pre_df.loc['annual']).iloc[-2:,:].T.set_index('symbol') ## .reset_index()
annual_df = pd.json_normalize(annual_pre_df['series']).reset_index()

quarterly_pre_df = pd.DataFrame(annual_and_quarterly_pre_df[annual_and_quarterly_pre_df['index'] == 'quarterly']).iloc[-2:,:].set_index('symbol') ## .reset_index()
## quarterly_pre_df = pd.DataFrame(annual_and_quarterly_pre_df.loc['quarterly']).iloc[-2:,:].T.set_index('symbol') ## .reset_index()
quarterly_df = pd.json_normalize(quarterly_pre_df['series']).reset_index()

df = df.drop(['annual','quarterly'], axis=0)
df = df.drop(columns=['metricType','series']).reset_index()

quarterly_df.head()

,symbol,assetTurnoverTTM,bookValue,cashRatio,currentRatio,ebitPerShare,eps,ev,evEbitdaTTM,evRevenueTTM,fcfMargin,fcfPerShareTTM,grossMargin,inventoryTurnoverTTM,longtermDebtTotalAsset,longtermDebtTotalCapital,longtermDebtTotalEquity,netDebtToTotalCapital,netDebtToTotalEquity,netMargin,operatingMargin,payoutRatioTTM,pb,peTTM,pfcfTTM,pretaxMargin,psTTM,ptbv,quickRatio,receivablesTurnoverTTM,roaTTM,roeTTM,roicTTM,rotcTTM,salesPerShare,sgaToSale,tangibleBookValue,totalDebtToEquity,totalDebtToTotalAsset,totalDebtToTotalCapital,totalRatio
0,AAPL,"[{'period': '2025-12-27', 'v': 1.2435}, {'peri...","[{'period': '2025-12-27', 'v': 88190}, {'perio...","[{'period': '2025-12-27', 'v': 0.2791022806358...","[{'period': '2025-12-27', 'v': 0.9737}, {'peri...","[{'period': '2025-12-27', 'v': 3.4335}, {'peri...","[{'period': '2025-12-27', 'v': 2.8424}, {'peri...","[{'period': '2025-12-27', 'v': 4090366.2}, {'p...","[{'period': '2025-12-27', 'v': 26.7379}, {'per...","[{'period': '2025-12-27', 'v': 9.3898}, {'peri...","[{'period': '2025-12-27', 'v': 0.3586}, {'peri...","[{'period': '2025-12-27', 'v': 8.3878}, {'peri...","[{'period': '2025-12-27', 'v': 0.4816}, {'peri...","[{'period': '2025-12-27', 'v': 35.8924}, {'per...","[{'period': '2025-12-27', 'v': 0.2022}, {'peri...","[{'period': '2025-12-27', 'v': 0.4291}, {'peri...","[{'period': '2025-12-27', 'v': 0.8695}, {'peri...","[{'period': '2025-12-27', 'v': 0.2529}, {'peri...","[{'period': '2025-12-27', 'v': 0.5124}, {'peri...","[{'period': '2025-12-27', 'v': 0.2928}, {'peri...","[{'period': '2025-12-27', 'v': 0.3537}, {'peri...","[{'period': '2025-12-27', 'v': 0.1315}, {'peri...","[{'period': '2025-12-27', 'v': 45.8689}, {'per...","[{'period': '2025-12-27', 'v': 34.346}, {'peri...","[{'period': '2025-12-27', 'v': 32.8012}, {'per...","[{'period': '2025-12-27', 'v': 0.3548}, {'peri...","[{'period': '2025-12-27', 'v': 9.2861}, {'peri...","[{'period': '2017-12-30', 'v': 6.4066}, {'peri...","[{'period': '2025-12-27', 'v': 0.9376}, {'peri...","[{'period': '2025-12-27', 'v': 12.5249}, {'per...","[{'period': '2025-12-27', 'v': 0.3362}, {'peri...","[{'period': '2025-12-27', 'v': 1.5994}, {'peri...","[{'period': '2025-12-27', 'v': 0.6892}, {'peri...","[{'period': '2025-12-27', 'v': 0.8255}, {'peri...","[{'period': '2025-12-27', 'v': 9.7065}, {'peri...","[{'period': '2025-12-27', 'v': 0.5184}, {'peri...","[{'period': '2017-12-30', 'v': 138050}, {'peri...","[{'period': '2025-12-27', 'v': 1.0263}, {'peri...","[{'period': '2025-12-27', 'v': 0.2386}, {'peri...","[{'period': '2025-12-27', 'v': 0.5065}, {'peri...","[{'period': '2025-12-27', 'v': 1.3029}, {'peri..."


In [49]:
duckdb.register("df_view", df)
res = duckdb.query("SELECT * FROM df_view WHERE index == '10DayAverageTradingVolume'").to_df()
res.head()

,index,metric,symbol
0,10DayAverageTradingVolume,45.06356,AAPL


In [50]:
## annual_json_parsed_dfs = normalize_json_dataframe(annual_df)
## duckdb.register("annual_json_parsed_df_view", annual_json_parsed_dfs)
duckdb.register("annual_df_view", annual_df)
res = duckdb.query("SELECT \
                   CAST(json_extract(unnest(bookValue), '$.period') AS DATE) AS bookValue_period, \
                   CAST(json_extract(unnest(bookValue), '$.v') AS FLOAT) AS bookValue_value, \
                   CAST(json_extract(unnest(cashRatio), '$.period') AS DATE) AS cashRatio_period, \
                   CAST(json_extract(unnest(cashRatio), '$.v') AS FLOAT) AS cashRatio_value, \
                   CAST(json_extract(unnest(currentRatio), '$.period') AS DATE) AS currentRatio_period, \
                   CAST(json_extract(unnest(currentRatio), '$.v') AS FLOAT) AS currentRatio_value, \
                   CAST(json_extract(unnest(ebitPerShare), '$.period') AS DATE) AS ebitPerShare_period, \
                   CAST(json_extract(unnest(ebitPerShare), '$.v') AS FLOAT) AS ebitPerShare_value, \
                   CAST(json_extract(unnest(eps), '$.period') AS DATE) AS eps_period, \
                   CAST(json_extract(unnest(eps), '$.v') AS FLOAT) AS eps_value, \
                   CAST(json_extract(unnest(ev), '$.period') AS DATE) AS ev_period, \
                   CAST(json_extract(unnest(ev), '$.v') AS FLOAT) AS ev_value, \
                   CAST(json_extract(unnest(evEbitda), '$.period') AS DATE) AS evEbitda_period, \
                   CAST(json_extract(unnest(evEbitda), '$.v') AS FLOAT) AS evEbitda_value, \
                   CAST(json_extract(unnest(evRevenue), '$.period') AS DATE) AS evRevenue_period, \
                   CAST(json_extract(unnest(evRevenue), '$.v') AS FLOAT) AS evRevenue_value, \
                   CAST(json_extract(unnest(fcfMargin), '$.period') AS DATE) AS fcfMargin_period, \
                   CAST(json_extract(unnest(fcfMargin), '$.v') AS FLOAT) AS fcfMargin_value, \
                   CAST(json_extract(unnest(grossMargin), '$.period') AS DATE) AS grossMargin_period, \
                   CAST(json_extract(unnest(grossMargin), '$.v') AS FLOAT) AS grossMargin_value, \
                   CAST(json_extract(unnest(inventoryTurnover), '$.period') AS DATE) AS inventoryTurnover_period, \
                   CAST(json_extract(unnest(inventoryTurnover), '$.v') AS FLOAT) AS inventoryTurnover_value, \
                   CAST(json_extract(unnest(longtermDebtTotalAsset), '$.period') AS DATE) AS longtermDebtTotalAsset_period, \
                   CAST(json_extract(unnest(longtermDebtTotalAsset), '$.v') AS FLOAT) AS longtermDebtTotalAsset_value, \
                   CAST(json_extract(unnest(longtermDebtTotalCapital), '$.period') AS DATE) AS longtermDebtTotalCapital_period, \
                   CAST(json_extract(unnest(longtermDebtTotalCapital), '$.v') AS FLOAT) AS longtermDebtTotalCapital_value, \
                   CAST(json_extract(unnest(longtermDebtTotalEquity), '$.period') AS DATE) AS longtermDebtTotalEquity_period, \
                   CAST(json_extract(unnest(longtermDebtTotalEquity), '$.v') AS FLOAT) AS longtermDebtTotalEquity_value, \
                   CAST(json_extract(unnest(netDebtToTotalCapital), '$.period') AS DATE) AS netDebtToTotalCapital_period, \
                   CAST(json_extract(unnest(netDebtToTotalCapital), '$.v') AS FLOAT) AS netDebtToTotalCapital_value, \
                   CAST(json_extract(unnest(netDebtToTotalEquity), '$.period') AS DATE) AS netDebtToTotalEquity_period, \
                   CAST(json_extract(unnest(netDebtToTotalEquity), '$.v') AS FLOAT) AS netDebtToTotalEquity_value, \
                   CAST(json_extract(unnest(netMargin), '$.period') AS DATE) AS netMargin_period, \
                   CAST(json_extract(unnest(netMargin), '$.v') AS FLOAT) AS netMargin_value, \
                   CAST(json_extract(unnest(operatingMargin), '$.period') AS DATE) AS operatingMargin_period, \
                   CAST(json_extract(unnest(operatingMargin), '$.v') AS FLOAT) AS operatingMargin_value, \
                   CAST(json_extract(unnest(payoutRatio), '$.period') AS DATE) AS payoutRatio_period, \
                   CAST(json_extract(unnest(payoutRatio), '$.v') AS FLOAT) AS payoutRatio_value, \
                   CAST(json_extract(unnest(pb), '$.period') AS DATE) AS pb_period, \
                   CAST(json_extract(unnest(pb), '$.v') AS FLOAT) AS pb_value, \
                   CAST(json_extract(unnest(pe), '$.period') AS DATE) AS pe_period, \
                   CAST(json_extract(unnest(pe), '$.v') AS FLOAT) AS pe_value, \
                   CAST(json_extract(unnest(pfcf), '$.period') AS DATE) AS pfcf_period, \
                   CAST(json_extract(unnest(pfcf), '$.v') AS FLOAT) AS pfcf_value, \
                   CAST(json_extract(unnest(pretaxMargin), '$.period') AS DATE) AS pretaxMargin_period, \
                   CAST(json_extract(unnest(pretaxMargin), '$.v') AS FLOAT) AS pretaxMargin_value, \
                   CAST(json_extract(unnest(ps), '$.period') AS DATE) AS ps_period, \
                   CAST(json_extract(unnest(ps), '$.v') AS FLOAT) AS ps_value, \
                   CAST(json_extract(unnest(ptbv), '$.period') AS DATE) AS ptbv_period, \
                   CAST(json_extract(unnest(ptbv), '$.v') AS FLOAT) AS ptbv_value, \
                   CAST(json_extract(unnest(quickRatio), '$.period') AS DATE) AS quickRatio_period, \
                   CAST(json_extract(unnest(quickRatio), '$.v') AS FLOAT) AS quickRatio_value, \
                   CAST(json_extract(unnest(receivablesTurnover), '$.period') AS DATE) AS receivablesTurnover_period, \
                   CAST(json_extract(unnest(receivablesTurnover), '$.v') AS FLOAT) AS receivablesTurnover_value, \
                   CAST(json_extract(unnest(roa), '$.period') AS DATE) AS roa_period, \
                   CAST(json_extract(unnest(roa), '$.v') AS FLOAT) AS roa_value, \
                   CAST(json_extract(unnest(roe), '$.period') AS DATE) AS roe_period, \
                   CAST(json_extract(unnest(roe), '$.v') AS FLOAT) AS roe_value, \
                   CAST(json_extract(unnest(roic), '$.period') AS DATE) AS roic_period, \
                   CAST(json_extract(unnest(roic), '$.v') AS FLOAT) AS roic_value, \
                   CAST(json_extract(unnest(rotc), '$.period') AS DATE) AS rotc_period, \
                   CAST(json_extract(unnest(rotc), '$.v') AS FLOAT) AS rotc_value, \
                   CAST(json_extract(unnest(salesPerShare), '$.period') AS DATE) AS salesPerShare_period, \
                   CAST(json_extract(unnest(salesPerShare), '$.v') AS FLOAT) AS salesPerShare_value, \
                   CAST(json_extract(unnest(sgaToSale), '$.period') AS DATE) AS sgaToSale_period, \
                   CAST(json_extract(unnest(sgaToSale), '$.v') AS FLOAT) AS sgaToSale_value, \
                   CAST(json_extract(unnest(tangibleBookValue), '$.period') AS DATE) AS tangibleBookValue_period, \
                   CAST(json_extract(unnest(tangibleBookValue), '$.v') AS FLOAT) AS tangibleBookValue_value, \
                   CAST(json_extract(unnest(totalDebtToEquity), '$.period') AS DATE) AS totalDebtToEquity_period, \
                   CAST(json_extract(unnest(totalDebtToEquity), '$.v') AS FLOAT) AS totalDebtToEquity_value, \
                   CAST(json_extract(unnest(totalDebtToTotalAsset), '$.period') AS DATE) AS totalDebtToTotalAsset_period, \
                   CAST(json_extract(unnest(totalDebtToTotalAsset), '$.v') AS FLOAT) AS totalDebtToTotalAsset_value, \
                   CAST(json_extract(unnest(totalDebtToTotalCapital), '$.period') AS DATE) AS totalDebtToTotalCapital_period, \
                   CAST(json_extract(unnest(totalDebtToTotalCapital), '$.v') AS FLOAT) AS totalDebtToTotalCapital_value, \
                   CAST(json_extract(unnest(totalRatio), '$.period') AS DATE) AS totalRatio_period, \
                   CAST(json_extract(unnest(totalRatio), '$.v') AS FLOAT) AS totalRatio_value, \
                   symbol FROM annual_df_view").to_df()  ##WHERE period == '1985-09-27'").to_df()
res.head()


,bookValue_period,bookValue_value,cashRatio_period,cashRatio_value,currentRatio_period,currentRatio_value,ebitPerShare_period,ebitPerShare_value,eps_period,eps_value,ev_period,ev_value,evEbitda_period,evEbitda_value,evRevenue_period,evRevenue_value,fcfMargin_period,fcfMargin_value,grossMargin_period,grossMargin_value,inventoryTurnover_period,inventoryTurnover_value,longtermDebtTotalAsset_period,longtermDebtTotalAsset_value,longtermDebtTotalCapital_period,longtermDebtTotalCapital_value,longtermDebtTotalEquity_period,longtermDebtTotalEquity_value,netDebtToTotalCapital_period,netDebtToTotalCapital_value,netDebtToTotalEquity_period,netDebtToTotalEquity_value,netMargin_period,netMargin_value,operatingMargin_period,operatingMargin_value,payoutRatio_period,payoutRatio_value,pb_period,pb_value,pe_period,pe_value,pfcf_period,pfcf_value,pretaxMargin_period,pretaxMargin_value,ps_period,ps_value,ptbv_period,ptbv_value,quickRatio_period,quickRatio_value,receivablesTurnover_period,receivablesTurnover_value,roa_period,roa_value,roe_period,roe_value,roic_period,roic_value,rotc_period,rotc_value,salesPerShare_period,salesPerShare_value,sgaToSale_period,sgaToSale_value,tangibleBookValue_period,tangibleBookValue_value,totalDebtToEquity_period,totalDebtToEquity_value,totalDebtToTotalAsset_period,totalDebtToTotalAsset_value,totalDebtToTotalCapital_period,totalDebtToTotalCapital_value,totalRatio_period,totalRatio_value,symbol
0,2025-09-27,73733.0,2025-09-27,0.216952,2025-09-27,0.8933,2025-09-27,8.8672,2025-09-27,7.4650,2025-09-27,3822713.50,2025-09-27,26.468100,2025-09-27,9.1857,2025-09-27,0.2373,2025-09-27,0.4691,2025-09-27,33.983398,2025-09-27,0.2180,2025-09-27,0.4511,2025-09-27,1.0623,2025-09-27,0.3684,2025-09-27,0.8674,2025-09-27,0.2692,2025-09-27,0.3197,2025-09-27,0.1377,2025-09-27,50.978001,2025-09-27,33.557400,2025-09-27,38.056801,2025-09-27,0.3189,2025-09-27,9.0320,2016-09-24,4.8643,2025-09-27,0.8588,2025-09-27,11.3725,2025-09-27,0.3118,2025-09-27,1.5191,2025-09-27,0.6451,2025-09-27,0.7663,2025-09-27,27.735399,2025-09-27,0.5309,2016-09-24,125043.0,2025-09-27,1.3547,2025-09-27,0.2781,2025-09-27,0.5753,2025-09-27,1.2583,AAPL
1,2024-09-28,56950.0,2024-09-28,0.169753,2024-09-28,0.8673,2024-09-28,7.9968,2024-09-28,6.0836,2024-09-28,3599793.25,2024-09-28,26.679001,2024-09-28,9.2058,2024-09-28,0.2783,2024-09-28,0.4621,2024-09-28,30.895500,2024-09-28,0.2349,2024-09-28,0.5214,2024-09-28,1.5057,2024-09-28,0.4717,2024-09-28,1.3623,2024-09-28,0.2397,2024-09-28,0.3151,2024-09-28,0.1625,2024-09-28,61.847401,2024-09-28,37.575901,2024-09-28,32.371201,2024-09-28,0.3158,2024-09-28,9.0074,2015-09-26,5.4327,2024-09-28,0.8260,2024-09-28,12.4300,2024-09-28,0.2568,2024-09-28,1.6459,2024-09-28,0.5699,2024-09-28,0.7491,2024-09-28,25.378500,2024-09-28,0.5379,2015-09-26,115462.0,2024-09-28,1.8881,2024-09-28,0.2946,2024-09-28,0.6537,2024-09-28,1.1849,AAPL
2,2023-09-30,62146.0,2023-09-30,0.206217,2023-09-30,0.9880,2023-09-30,7.2285,2023-09-30,6.1341,2023-09-30,2783970.00,2023-09-30,22.194000,2023-09-30,7.2634,2023-09-30,0.2598,2023-09-30,0.4413,2023-09-30,37.977699,2023-09-30,0.2702,2023-09-30,0.5468,2023-09-30,1.5332,2023-09-30,0.4714,2023-09-30,1.3218,2023-09-30,0.2531,2023-09-30,0.2982,2023-09-30,0.1549,2023-09-30,43.475399,2023-09-30,27.855301,2023-09-30,27.131100,2023-09-30,0.2967,2023-09-30,7.0491,2014-09-27,5.4677,2023-09-30,0.9444,2023-09-30,13.2873,2023-09-30,0.2751,2023-09-30,1.5608,2023-09-30,0.5566,2023-09-30,0.6559,2023-09-30,24.239300,2023-09-30,0.5587,2014-09-27,107405.0,2023-09-30,1.8040,2023-09-30,0.3180,2023-09-30,0.6434,2023-09-30,1.2140,AAPL
3,2022-09-24,50672.0,2022-09-24,0.153563,2022-09-24,0.8794,2022-09-24,7.3158,2022-09-24,6.1132,2022-09-24,2501154.25,2022-09-24,19.193399,2022-09-24,6.3428,2022-09-24,0.2826,2022-09-24,0.4331,2022-09-24,38.789902,2022-09-24,0.2805,2022-09-24,0.5764,2022-09-24,1.9529,2022-09-24,0.5671,2022-09-24,1.9215,2022-09-24,0.2531,2022-09-24,0.3029,2022-09-24,0.1487,2022-09-24,47.438202,2022-09-24,24.085400,2022-09-24,21.56

In [51]:
## quarterly_json_parsed_dfs = normalize_json_dataframe(quarterly_df)
## duckdb.register("quarterly_json_parsed_df_view", quarterly_json_parsed_dfs)
duckdb.register("quarterly_df_view", quarterly_df)
res = duckdb.query("SELECT \
                   CAST(json_extract(unnest(assetTurnoverTTM), '$.period') AS DATE) AS assetTurnoverTTM_period, \
                   CAST(json_extract(unnest(assetTurnoverTTM), '$.v') AS FLOAT) AS assetTurnoverTTM_value, \
                   CAST(json_extract(unnest(bookValue), '$.period') AS DATE) AS bookValue_period, \
                   CAST(json_extract(unnest(bookValue), '$.v') AS FLOAT) AS bookValue_value, \
                   CAST(json_extract(unnest(cashRatio), '$.period') AS DATE) AS cashRatio_period, \
                   CAST(json_extract(unnest(cashRatio), '$.v') AS FLOAT) AS cashRatio_value, \
                   CAST(json_extract(unnest(currentRatio), '$.period') AS DATE) AS currentRatio_period, \
                   CAST(json_extract(unnest(currentRatio), '$.v') AS FLOAT) AS currentRatio_value, \
                   CAST(json_extract(unnest(ebitPerShare), '$.period') AS DATE) AS ebitPerShare_period, \
                   CAST(json_extract(unnest(ebitPerShare), '$.v') AS FLOAT) AS ebitPerShare_value, \
                   CAST(json_extract(unnest(eps), '$.period') AS DATE) AS eps_period, \
                   CAST(json_extract(unnest(eps), '$.v') AS FLOAT) AS eps_value, \
                   CAST(json_extract(unnest(ev), '$.period') AS DATE) AS ev_period, \
                   CAST(json_extract(unnest(ev), '$.v') AS FLOAT) AS ev_value, \
                   CAST(json_extract(unnest(evEbitdaTTM), '$.period') AS DATE) AS evEbitdaTTM_period, \
                   CAST(json_extract(unnest(evEbitdaTTM), '$.v') AS FLOAT) AS evEbitdaTTM_value, \
                   CAST(json_extract(unnest(evRevenueTTM), '$.period') AS DATE) AS evRevenueTTM_period, \
                   CAST(json_extract(unnest(evRevenueTTM), '$.v') AS FLOAT) AS evRevenueTTM_value, \
                   CAST(json_extract(unnest(fcfMargin), '$.period') AS DATE) AS fcfMargin_period, \
                   CAST(json_extract(unnest(fcfMargin), '$.v') AS FLOAT) AS fcfMargin_value, \
                   CAST(json_extract(unnest(fcfPerShareTTM), '$.period') AS DATE) AS fcfPerShareTTM_period, \
                   CAST(json_extract(unnest(fcfPerShareTTM), '$.v') AS FLOAT) AS fcfPerShareTTM_value, \
                   CAST(json_extract(unnest(grossMargin), '$.period') AS DATE) AS grossMargin_period, \
                   CAST(json_extract(unnest(grossMargin), '$.v') AS FLOAT) AS grossMargin_value, \
                   CAST(json_extract(unnest(inventoryTurnoverTTM), '$.period') AS DATE) AS inventoryTurnoverTTM_period, \
                   CAST(json_extract(unnest(inventoryTurnoverTTM), '$.v') AS FLOAT) AS inventoryTurnoverTTM_value, \
                   CAST(json_extract(unnest(longtermDebtTotalAsset), '$.period') AS DATE) AS longtermDebtTotalAsset_period, \
                   CAST(json_extract(unnest(longtermDebtTotalAsset), '$.v') AS FLOAT) AS longtermDebtTotalAsset_value, \
                   CAST(json_extract(unnest(longtermDebtTotalCapital), '$.period') AS DATE) AS longtermDebtTotalCapital_period, \
                   CAST(json_extract(unnest(longtermDebtTotalCapital), '$.v') AS FLOAT) AS longtermDebtTotalCapital_value, \
                   CAST(json_extract(unnest(longtermDebtTotalEquity), '$.period') AS DATE) AS longtermDebtTotalEquity_period, \
                   CAST(json_extract(unnest(longtermDebtTotalEquity), '$.v') AS FLOAT) AS longtermDebtTotalEquity_value, \
                   CAST(json_extract(unnest(netDebtToTotalCapital), '$.period') AS DATE) AS netDebtToTotalCapital_period, \
                   CAST(json_extract(unnest(netDebtToTotalCapital), '$.v') AS FLOAT) AS netDebtToTotalCapital_value, \
                   CAST(json_extract(unnest(netDebtToTotalEquity), '$.period') AS DATE) AS netDebtToTotalEquity_period, \
                   CAST(json_extract(unnest(netDebtToTotalEquity), '$.v') AS FLOAT) AS netDebtToTotalEquity_value, \
                   CAST(json_extract(unnest(netMargin), '$.period') AS DATE) AS netMargin_period, \
                   CAST(json_extract(unnest(netMargin), '$.v') AS FLOAT) AS netMargin_value, \
                   CAST(json_extract(unnest(operatingMargin), '$.period') AS DATE) AS operatingMargin_period, \
                   CAST(json_extract(unnest(operatingMargin), '$.v') AS FLOAT) AS operatingMargin_value, \
                   CAST(json_extract(unnest(payoutRatioTTM), '$.period') AS DATE) AS payoutRatioTTM_period, \
                   CAST(json_extract(unnest(payoutRatioTTM), '$.v') AS FLOAT) AS payoutRatioTTM_value, \
                   CAST(json_extract(unnest(pb), '$.period') AS DATE) AS pb_period, \
                   CAST(json_extract(unnest(pb), '$.v') AS FLOAT) AS pb_value, \
                   CAST(json_extract(unnest(peTTM), '$.period') AS DATE) AS peTTM_period, \
                   CAST(json_extract(unnest(peTTM), '$.v') AS FLOAT) AS peTTM_value, \
                   CAST(json_extract(unnest(pfcfTTM), '$.period') AS DATE) AS pfcfTTM_period, \
                   CAST(json_extract(unnest(pfcfTTM), '$.v') AS FLOAT) AS pfcfTTM_value, \
                   CAST(json_extract(unnest(pretaxMargin), '$.period') AS DATE) AS pretaxMargin_period, \
                   CAST(json_extract(unnest(pretaxMargin), '$.v') AS FLOAT) AS pretaxMargin_value, \
                   CAST(json_extract(unnest(psTTM), '$.period') AS DATE) AS psTTM_period, \
                   CAST(json_extract(unnest(psTTM), '$.v') AS FLOAT) AS psTTM_value, \
                   CAST(json_extract(unnest(ptbv), '$.period') AS DATE) AS ptbv_period, \
                   CAST(json_extract(unnest(ptbv), '$.v') AS FLOAT) AS ptbv_value, \
                   CAST(json_extract(unnest(quickRatio), '$.period') AS DATE) AS quickRatio_period, \
                   CAST(json_extract(unnest(quickRatio), '$.v') AS FLOAT) AS quickRatio_value, \
                   CAST(json_extract(unnest(receivablesTurnoverTTM), '$.period') AS DATE) AS receivablesTurnoverTTM_period, \
                   CAST(json_extract(unnest(receivablesTurnoverTTM), '$.v') AS FLOAT) AS receivablesTurnoverTTM_value, \
                   CAST(json_extract(unnest(roaTTM), '$.period') AS DATE) AS roaTTM_period, \
                   CAST(json_extract(unnest(roaTTM), '$.v') AS FLOAT) AS roaTTM_value, \
                   CAST(json_extract(unnest(roeTTM), '$.period') AS DATE) AS roeTTM_period, \
                   CAST(json_extract(unnest(roeTTM), '$.v') AS FLOAT) AS roeTTM_value, \
                   CAST(json_extract(unnest(roicTTM), '$.period') AS DATE) AS roicTTM_period, \
                   CAST(json_extract(unnest(roicTTM), '$.v') AS FLOAT) AS roicTTM_value, \
                   CAST(json_extract(unnest(rotcTTM), '$.period') AS DATE) AS rotcTTM_period, \
                   CAST(json_extract(unnest(rotcTTM), '$.v') AS FLOAT) AS rotcTTM_value, \
                   CAST(json_extract(unnest(salesPerShare), '$.period') AS DATE) AS salesPerShare_period, \
                   CAST(json_extract(unnest(salesPerShare), '$.v') AS FLOAT) AS salesPerShare_value, \
                   CAST(json_extract(unnest(sgaToSale), '$.period') AS DATE) AS sgaToSale_period, \
                   CAST(json_extract(unnest(sgaToSale), '$.v') AS FLOAT) AS sgaToSale_value, \
                   CAST(json_extract(unnest(tangibleBookValue), '$.period') AS DATE) AS tangibleBookValue_period, \
                   CAST(json_extract(unnest(tangibleBookValue), '$.v') AS FLOAT) AS tangibleBookValue_value, \
                   CAST(json_extract(unnest(totalDebtToEquity), '$.period') AS DATE) AS totalDebtToEquity_period, \
                   CAST(json_extract(unnest(totalDebtToEquity), '$.v') AS FLOAT) AS totalDebtToEquity_value, \
                   CAST(json_extract(unnest(totalDebtToTotalAsset), '$.period') AS DATE) AS totalDebtToTotalAsset_period, \
                   CAST(json_extract(unnest(totalDebtToTotalAsset), '$.v') AS FLOAT) AS totalDebtToTotalAsset_value, \
                   CAST(json_extract(unnest(totalDebtToTotalCapital), '$.period') AS DATE) AS totalDebtToTotalCapital_period, \
                   CAST(json_extract(unnest(totalDebtToTotalCapital), '$.v') AS FLOAT) AS totalDebtToTotalCapital_value, \
                   CAST(json_extract(unnest(totalRatio), '$.period') AS DATE) AS totalRatio_period, \
                   CAST(json_extract(unnest(totalRatio), '$.v') AS FLOAT) AS totalRatio_value, \
                   symbol FROM quarterly_df_view").to_df()  ##WHERE period == '1989-06-30'").to_df()
res.head()

,assetTurnoverTTM_period,assetTurnoverTTM_value,bookValue_period,bookValue_value,cashRatio_period,cashRatio_value,currentRatio_period,currentRatio_value,ebitPerShare_period,ebitPerShare_value,eps_period,eps_value,ev_period,ev_value,evEbitdaTTM_period,evEbitdaTTM_value,evRevenueTTM_period,evRevenueTTM_value,fcfMargin_period,fcfMargin_value,fcfPerShareTTM_period,fcfPerShareTTM_value,grossMargin_period,grossMargin_value,inventoryTurnoverTTM_period,inventoryTurnoverTTM_value,longtermDebtTotalAsset_period,longtermDebtTotalAsset_value,longtermDebtTotalCapital_period,longtermDebtTotalCapital_value,longtermDebtTotalEquity_period,longtermDebtTotalEquity_value,netDebtToTotalCapital_period,netDebtToTotalCapital_value,netDebtToTotalEquity_period,netDebtToTotalEquity_value,netMargin_period,netMargin_value,operatingMargin_period,operatingMargin_value,payoutRatioTTM_period,payoutRatioTTM_value,pb_period,pb_value,peTTM_period,peTTM_value,pfcfTTM_period,pfcfTTM_value,pretaxMargin_period,pretaxMargin_value,psTTM_period,psTTM_value,ptbv_period,ptbv_value,quickRatio_period,quickRatio_value,receivablesTurnoverTTM_period,receivablesTurnoverTTM_value,roaTTM_period,roaTTM_value,roeTTM_period,roeTTM_value,roicTTM_period,roicTTM_value,rotcTTM_period,rotcTTM_value,salesPerShare_period,salesPerShare_value,sgaToSale_period,sgaToSale_value,tangibleBookValue_period,tangibleBookValue_value,totalDebtToEquity_period,totalDebtToEquity_value,totalDebtToTotalAsset_period,totalDebtToTotalAsset_value,totalDebtToTotalCapital_period,totalDebtToTotalCapital_value,totalRatio_period,totalRatio_value,symbol
0,2025-12-27,1.2435,2025-12-27,88190.0,2025-12-27,0.279102,2025-12-27,0.9737,2025-12-27,3.4335,2025-12-27,2.8424,2025-12-27,4090366.25,2025-12-27,26.7379,2025-12-27,9.3898,2025-12-27,0.3586,2025-12-27,8.3878,2025-12-27,0.4816,2025-12-27,35.892399,2025-12-27,0.2022,2025-12-27,0.4291,2025-12-27,0.8695,2025-12-27,0.2529,2025-12-27,0.5124,2025-12-27,0.2928,2025-12-27,0.3537,2025-12-27,0.1315,2025-12-27,45.868900,2025-12-27,34.346001,2025-12-27,32.801201,2025-12-27,0.3548,2025-12-27,9.2861,2017-12-30,6.4066,2025-12-27,0.9376,2025-12-27,12.524900,2025-12-27,0.3362,2025-12-27,1.5994,2025-12-27,0.6892,2025-12-27,0.8255,2025-12-27,9.7065,2025-12-27,0.5184,2017-12-30,138050.0,2025-12-27,1.0263,2025-12-27,0.2386,2025-12-27,0.5065,2025-12-27,1.3029,AAPL
1,2025-09-27,1.2186,2025-09-27,73733.0,2025-09-27,0.216952,2025-09-27,0.8933,2025-09-27,2.1816,2025-09-27,1.8479,2025-09-27,3821483.50,2025-09-27,26.4594,2025-09-27,9.1827,2025-09-27,0.2585,2025-09-27,6.6855,2025-09-27,0.4718,2025-09-27,33.983398,2025-09-27,0.2180,2025-09-27,0.4544,2025-09-27,1.0623,2025-09-27,0.3638,2025-09-27,0.8507,2025-09-27,0.2680,2025-09-27,0.3165,2025-09-27,0.1377,2025-09-27,50.978001,2025-09-27,33.557400,2025-09-27,38.056801,2025-09-27,0.3201,2025-09-27,9.0320,2017-07-01,5.7561,2025-09-27,0.8588,2025-09-27,11.372500,2025-09-27,0.3280,2025-09-27,1.6405,2025-09-27,0.6703,2025-09-27,0.7962,2025-09-27,6.8937,2025-09-27,0.5282,2017-07-01,129981.0,2025-09-27,1.3380,2025-09-27,0.2746,2025-09-27,0.5723,2025-09-27,1.2583,AAPL
2,2025-06-28,1.1915,2025-06-28,65830.0,2025-06-28,0.257008,2025-06-28,0.8680,2025-06-28,1.8867,2025-06-28,1.5677,2025-06-28,3113582.75,2025-06-28,22.0793,2025-06-28,7.6197,2025-06-28,0.2595,2025-06-28,6.4741,2025-06-28,0.4649,2025-06-28,36.043999,2025-06-28,0.2487,2025-06-28,0.4920,2025-06-28,1.2522,2025-06-28,0.3906,2025-06-28,0.9939,2025-06-28,0.2492,2025-06-28,0.2999,2025-06-28,0.1547,2025-06-28,46.303398,2025-06-28,30.702600,2025-06-28,31.690901,2025-06-28,0.2981,2025-06-28,7.4595,2017-04-01,5.6903,2025-06-28,0.8260,2025-06-28,16.230700,2025-06-28,0.2895,2025-06-28,1.5492,2025-06-28,0.6020,2025-06-28,0.7896,2025-06-28,6.2908,2025-06-28,0.5351,2017-04-01,131465.0,2025-06-28,1.5449,2025-06-28,0.3068,2025-06-28,0.6071,2025-06-28,1.2478,AAPL
3,2025-03-29,1.1673,2025-03-29,66796.0,2025-03-29,0.194797,2025-03-29,0.8209,2025-03-29,1.9652,2025-03-29,1.6458,2025-03-29,3388494.00,2025-03-29,24.4657,

##### Earnings Surprises

In [52]:
# Earnings surprises
## print(finnhub_client.company_earnings(test_symbol, limit=5))
df = pd.DataFrame(finnhub_client.company_earnings(test_symbol))
df.head()

,actual,estimate,period,quarter,surprise,surprisePercent,symbol,year
0,2.84,2.7257,2025-12-31,1,0.1143,4.1934,AAPL,2026
1,1.85,1.8075,2025-09-30,4,0.0425,2.3513,AAPL,2025
2,1.57,1.4626,2025-06-30,3,0.1074,7.3431,AAPL,2025
3,1.65,1.6596,2025-03-31,2,-0.0096,-0.5785,AAPL,2025


##### Company News

In [53]:
# Company News
# Need to use _from instead of from to avoid conflict
df = pd.json_normalize(finnhub_client.company_news(test_symbol, _from=pd.to_datetime("today").date(), to=pd.to_datetime("today").date()))
df['datetime_converted'] = pd.to_datetime(df['datetime'], unit='s')
df.drop(columns=['image'], inplace=True)
df.head()

,category,datetime,headline,id,related,source,summary,url,datetime_converted
0,company,1775072494,Apple Inc. (AAPL) Expands U.S. Manufacturing P...,139602863,AAPL,Yahoo,Apple Inc. (NASDAQ:AAPL) is one of Motley Fool...,https://finnhub.io/api/news?id=34646f1607731e0...,2026-04-01 19:41:34
1,company,1775070114,Apple Inc. (AAPL): D. E. Shaw Trims Holding,139602864,AAPL,Yahoo,Apple Inc. (NASDAQ:AAPL) features on the D. E....,https://finnhub.io/api/news?id=b394667fc7050b2...,2026-04-01 19:01:54
2,company,1775065320,Babel Street Extends Free Real-Time Risk Intel...,139602017,AAPL,Yahoo,"WASHINGTON, April 01, 2026--Babel Street, a gl...",https://finnhub.io/api/news?id=e39acc5a7500a9d...,2026-04-01 17:42:00
3,company,1775063520,Apple at 50: Tech giant shows no signs of slow...,139602038,AAPL,Yahoo,"Apple Inc (NASDAQ:AAPL, XETRA:APC) is celebrat...",https://finnhub.io/api/news?id=f573953af9551c1...,2026-04-01 17:12:00
4,company,1775062431,What's Next For Apple After Company's 50th Ann...,139602039,AAPL,Yahoo,"After celebrating its 50th anniversary, Apple ...",https://finnhub.io/api/news?id=2d60ad839fb409f...,2026-04-01 16:53:51


##### Company Peers

In [11]:
# Company Peers
df = pd.DataFrame(finnhub_client.company_peers(test_symbol))
df['symbol'] = test_symbol
df = df[df[0]!=df['symbol']]
df.rename(columns={0:'peer'}, inplace=True)
df.head()

,peer,symbol
1,DELL,AAPL
2,SNDK,AAPL
3,WDC,AAPL
4,HPE,AAPL
5,PSTG,AAPL


##### Company Profile 2

In [55]:
# Company Profile 2
df = pd.json_normalize(finnhub_client.company_profile2(symbol=test_symbol))
df.head()

,country,currency,estimateCurrency,exchange,finnhubIndustry,ipo,logo,marketCapitalization,name,phone,shareOutstanding,ticker,weburl
0,US,USD,USD,NASDAQ NMS - GLOBAL MARKET,Technology,1980-12-12,https://static2.finnhub.io/file/publicdatany/f...,3.725926e+06,Apple Inc,14089961010,14702.7,AAPL,https://www.apple.com/


##### List Country

In [56]:
# List country
df = pd.json_normalize(finnhub_client.country())
df.head()

,code2,code3,codeNo,country,countryRiskPremium,currency,currencyCode,defaultSpread,equityRiskPremium,rating,region,subRegion
0,NR,NRU,520,Nauru,NaN,Australian Dollars,AUD,None,NaN,NaN,Oceania,Micronesia
1,MF,MAF,663,Saint Martin,NaN,Netherlands Antillean guilder,ANG,None,NaN,NaN,Americas,Latin America and the Caribbean
2,GE,GEO,268,Georgia,4.40,Lari,GEL,None,9.00,Ba2,Asia,Western Asia
3,AQ,ATA,10,Antarctica,NaN,Antarctican dollar,AQD,None,NaN,NaN,,
4,VC,VCT,670,Saint Vincent and the Grenadines,9.51,East Caribbean Dollar,XCD,None,14.11,B3,Americas,Latin America and the Caribbean


##### Crypto Exchange

In [57]:
# Crypto Exchange
df = pd.DataFrame(finnhub_client.crypto_exchanges())
df.head()

,0
0,BINANCE
1,POLONIEX
2,OKEX
3,BINANCEUS
4,COINBASE


##### Crypto Symbols

In [58]:
# Crypto symbols
df = pd.json_normalize(finnhub_client.crypto_symbols('BINANCE'))
df.head()

,description,displaySymbol,symbol
0,Binance KSM/BUSD,KSM/BUSD,BINANCE:KSMBUSD
1,Binance TRX/BUSD,TRX/BUSD,BINANCE:TRXBUSD
2,Binance STRK/TRY,STRK/TRY,BINANCE:STRKTRY
3,Binance VIRTUAL/TRY,VIRTUAL/TRY,BINANCE:VIRTUALTRY
4,Binance USTC/TRY,USTC/TRY,BINANCE:USTCTRY


##### Financials Reported

In [2]:
# Financials as reported
df = pd.json_normalize(finnhub_client.financials_reported(symbol=test_symbol, freq='annual'))
## df = pd.json_normalize(df['data']).reset_index()
df = normalize_json_dataframe(pd.DataFrame(df['data']))
df = pd.DataFrame(df[0]).drop(columns=['original_index'])
## report_bs_df = normalize_json_dataframe(df)## pd.DataFrame(df['report.bs']))
## report_bs_df
df.head()

,accessNumber,symbol,cik,year,quarter,form,startDate,endDate,filedDate,acceptedDate,report.bs,report.ic,report.cf
0,0000320193-25-000079,AAPL,320193,2025,0,10-K,2024-09-29 00:00:00,2025-09-27 00:00:00,2025-10-31 00:00:00,2025-10-31 06:01:26,[{'concept': 'us-gaap_CashAndCashEquivalentsAt...,[{'concept': 'us-gaap_RevenueFromContractWithC...,"[{'concept': 'us-gaap_NetIncomeLoss', 'unit': ..."
1,0000320193-24-000123,AAPL,320193,2024,0,10-K,2023-10-01 00:00:00,2024-09-28 00:00:00,2024-11-01 00:00:00,2024-11-01 06:01:36,[{'concept': 'us-gaap_CashAndCashEquivalentsAt...,[{'concept': 'us-gaap_RevenueFromContractWithC...,"[{'concept': 'us-gaap_NetIncomeLoss', 'unit': ..."
2,0000320193-23-000106,AAPL,320193,2023,0,10-K,2022-09-25 00:00:00,2023-09-30 00:00:00,2023-11-03 00:00:00,2023-11-02 18:08:27,[{'concept': 'us-gaap_CashAndCashEquivalentsAt...,[{'concept': 'us-gaap_RevenueFromContractWithC...,"[{'concept': 'us-gaap_NetIncomeLoss', 'unit': ..."
3,0000320193-22-000108,AAPL,320193,2022,0,10-K,2021-09-26 00:00:00,2022-09-24 00:00:00,2022-10-28 00:00:00,2022-10-27 18:01:14,[{'concept': 'us-gaap_CashAndCashEquivalentsAt...,[{'concept': 'us-gaap_RevenueFromContractWithC...,"[{'concept': 'us-gaap_NetIncomeLoss', 'unit': ..."
4,0000320193-21-000105,AAPL,320193,2021,0,10-K,2020-09-27 00:00:00,2021-09-25 00:00:00,2021-10-29 00:00:00,2021-10-28 18:04:28,[{'concept': 'us-gaap_CashAndCashEquivalentsAt...,[{'concept': 'us-gaap_RevenueFromContractWithC...,"[{'concept': 'us-gaap_NetIncomeLoss', 'unit': ..."


In [6]:
report_pre_df = df.copy(deep=True)
report_pre_df.rename(columns={'report.bs':'bs', 'report.ic':'ic', 'report.cf':'cf'}, inplace=True)
duckdb.register("report_pre_df_view", report_pre_df)
res = duckdb.query("SELECT \
                   accessNumber, symbol, cik, year, quarter, form, startDate, endDate, filedDate, acceptedDate, \
                   CAST(json_extract(unnest(bs), '$.concept') AS STRING) AS report_bs_concept, \
                   CAST(json_extract(unnest(bs), '$.unit') AS STRING) AS report_bs_unit, \
                   CAST(json_extract(unnest(bs), '$.label') AS STRING) AS report_bs_label, \
                   CAST(json_extract(unnest(bs), '$.value') AS FLOAT) AS report_bs_value, \
                   CAST(json_extract(unnest(ic), '$.concept') AS STRING) AS report_ic_concept, \
                   CAST(json_extract(unnest(ic), '$.unit') AS STRING) AS report_ic_unit, \
                   CAST(json_extract(unnest(ic), '$.label') AS STRING) AS report_ic_label, \
                   CAST(json_extract(unnest(ic), '$.value') AS FLOAT) AS report_ic_value, \
                   CAST(json_extract(unnest(cf), '$.concept') AS STRING) AS report_cf_concept, \
                   CAST(json_extract(unnest(cf), '$.unit') AS STRING) AS report_cf_unit, \
                   CAST(json_extract(unnest(cf), '$.label') AS STRING) AS report_cf_label, \
                   CAST(json_extract(unnest(cf), '$.value') AS FLOAT) AS report_cf_value, \
                   FROM report_pre_df_view").to_df()  ##WHERE period == '1985-09-27'").to_df()
res.replace('"', '', regex=True, inplace=True)
res.head()

,accessNumber,symbol,cik,year,quarter,form,startDate,endDate,filedDate,acceptedDate,report_bs_concept,report_bs_unit,report_bs_label,report_bs_value,report_ic_concept,report_ic_unit,report_ic_label,report_ic_value,report_cf_concept,report_cf_unit,report_cf_label,report_cf_value
0,0000320193-25-000079,AAPL,320193,2025,0,10-K,2024-09-29 00:00:00,2025-09-27 00:00:00,2025-10-31 00:00:00,2025-10-31 06:01:26,us-gaap_CashAndCashEquivalentsAtCarryingValue,usd,Cash and cash equivalents,3.593400e+10,us-gaap_RevenueFromContractWithCustomerExcludi...,usd,Net sales,4.161610e+11,us-gaap_NetIncomeLoss,usd,Net income,1.120100e+11
1,0000320193-25-000079,AAPL,320193,2025,0,10-K,2024-09-29 00:00:00,2025-09-27 00:00:00,2025-10-31 00:00:00,2025-10-31 06:01:26,us-gaap_MarketableSecuritiesCurrent,usd,Marketable securities,1.876300e+10,us-gaap_CostOfGoodsAndServicesSold,usd,Cost of sales,2.209600e+11,us-gaap_DepreciationDepletionAndAmortization,usd,Depreciation and amortization,1.169800e+10
2,0000320193-25-000079,AAPL,320193,2025,0,10-K,2024-09-29 00:00:00,2025-09-27 00:00:00,2025-10-31 00:00:00,2025-10-31 06:01:26,us-gaap_AccountsReceivableNetCurrent,usd,"Accounts receivable, net",3.977700e+10,us-gaap_GrossProfit,usd,Gross margin,1.952010e+11,us-gaap_ShareBasedCompensation,usd,Share-based compensation expense,1.286300e+10
3,0000320193-25-000079,AAPL,320193,2025,0,10-K,2024-09-29 00:00:00,2025-09-27 00:00:00,2025-10-31 00:00:00,2025-10-31 06:01:26,us-gaap_NontradeReceivablesCurrent,usd,Vendor non-trade receivables,3.318000e+10,us-gaap_ResearchAndDevelopmentExpense,usd,Research and development,3.455000e+10,us-gaap_OtherNoncashIncomeExpense,usd,Other,8.900000e+07
4,0000320193-25-000079,AAPL,320193,2025,0,10-K,2024-09-29 00:00:00,2025-09-27 00:00:00,2025-10-31 00:00:00,2025-10-31 06:01:26,us-gaap_InventoryNet,usd,Inventories,5.718000e+09,us-gaap_SellingGeneralAndAdministrativeExpense,usd,"Selling, general and administrative",2.760100e+10,us-gaap_IncreaseDecreaseInAccountsReceivable,usd,"Accounts receivable, net",6.682000e+09


##### Forex Exchanges

In [10]:
# Forex exchanges
df = pd.DataFrame(finnhub_client.forex_exchanges())
df.rename(columns={0:'forex_exch'}, inplace=True)
df.head()

,forex_exch
0,oanda
1,fxcm
2,forex.com
3,fhfx
4,capital


##### Forex Symbols

In [12]:
# Forex symbols
df = pd.DataFrame(finnhub_client.forex_symbols('OANDA'))
df.head()

,description,displaySymbol,symbol
0,Oanda Gold/HKD,XAU/HKD,OANDA:XAU_HKD
1,Oanda Gold/SGD,XAU/SGD,OANDA:XAU_SGD
2,Oanda Gold/AUD,XAU/AUD,OANDA:XAU_AUD
3,Oanda EUR/SGD,EUR/SGD,OANDA:EUR_SGD
4,Oanda AUD/HKD,AUD/HKD,OANDA:AUD_HKD


##### General News

In [14]:
# General news
df = pd.DataFrame(finnhub_client.general_news('forex', min_id=0))
df.drop(columns=['image'], inplace=True)
df.head()

,category,datetime,headline,id,related,source,summary,url
0,forex,1775077327,"Trump to overhaul steel, aluminium tariffs, li...",7731259,,Forexlive,<p>US tariff overhaul simplifies rules but ris...,https://investinglive.com/news/trump-to-overha...


##### IPO Calendar

In [17]:
# IPO calendar
df = pd.DataFrame(finnhub_client.ipo_calendar(_from="2020-05-01", to="2020-06-01"))
duckdb.register("df_view", df)
res = duckdb.query("SELECT \
                   CAST(json_extract(ipoCalendar, '$.date') AS DATE) AS date, \
                   CAST(json_extract(ipoCalendar, '$.exchange') AS STRING) AS exchange, \
                   CAST(json_extract(ipoCalendar, '$.name') AS STRING) AS name, \
                   CAST(json_extract(ipoCalendar, '$.numberOfShares') AS INT) AS numberOfShares, \
                   CAST(json_extract(ipoCalendar, '$.price') AS FLOAT) AS price, \
                   CAST(json_extract(ipoCalendar, '$.status') AS STRING) AS status, \
                   CAST(json_extract(ipoCalendar, '$.symbol') AS STRING) AS symbol, \
                   CAST(json_extract(ipoCalendar, '$.totalSharesValue') AS FLOAT) AS totalSharesValue, \
                   FROM df_view").to_df()  ##WHERE period == '1985-09-27'").to_df()
res.replace('"', '', regex=True, inplace=True)
res.head()

,date,exchange,name,numberOfShares,price,status,symbol,totalSharesValue
0,2020-05-27,NYSE,Foley Trasimene Acquisition Corp.,90000000,10.0,priced,WPF'U,900000000.0
1,2020-05-22,NASDAQ Global Select,"Inari Medical, Inc.",8202565,19.0,priced,NARI,155848736.0
2,2020-05-21,NYSE,"Butterfly Network, Inc.",36000000,10.0,priced,LGVWU,360000000.0
3,2020-05-21,NYSE,"SelectQuote, Inc.",28500000,20.0,priced,SLQT,570000000.0
4,2020-05-20,NYSE,"Eos Energy Enterprises, Inc.",17500000,10.0,priced,BMRGU,175000000.0


##### Quote

In [2]:
# Quote
df = pd.json_normalize(finnhub_client.quote(test_symbol))
df

,c,d,dp,h,l,o,pc,t
0,255.63,1.84,0.725,256.18,253.33,254.08,253.79,1775073600


##### Recommendation Trends

In [4]:
# Recommendation trends
df = pd.json_normalize(finnhub_client.recommendation_trends(test_symbol))
df

,buy,hold,period,sell,strongBuy,strongSell,symbol
0,23,15,2026-04-01,2,14,0,AAPL
1,22,16,2026-03-01,2,14,0,AAPL
2,21,17,2026-02-01,2,14,0,AAPL
3,21,16,2026-01-01,2,14,0,AAPL


##### Stock Symbols

In [13]:
# Stock symbols
df = pd.json_normalize(finnhub_client.stock_symbols('US')[0:5])
df

,currency,description,displaySymbol,figi,figiComposite,isin,mic,shareClassFIGI,symbol,symbol2,type
0,USD,CAHYA MATA SARAWAK BHD,CHYMF,BBG00NTZXF73,BBG00NTZXF73,,OOTC,BBG001S6KDN7,CHYMF,,Common Stock
1,USD,BAHL & GAYNOR S/M INC GROWTH,SMIG,BBG0129P6L24,BBG0129P6L24,,ARCX,BBG0129P6LY9,SMIG,,ETP
2,USD,DOLBY LABORATORIES INC-CL A,DLB,BBG000DGLTG5,BBG000DGLTG5,,XNYS,BBG001SDXGH8,DLB,,Common Stock
3,USD,DIREXION DY CSCO BULL 2X ETF,CSCL,BBG01VF24WR5,BBG01VF24WR5,,XNAS,BBG01VF24Y37,CSCL,,ETP
4,USD,SWIRE PACIFIC LTD-CL B,SWRBF,BBG000BXQSZ6,BBG000BXQSZ6,,OOTC,BBG001S6DQ65,SWRBF,,Common Stock


##### Earnings Calendar

In [14]:
# Earnings Calendar
df = pd.json_normalize(finnhub_client.earnings_calendar(_from="2005-03-16", to="2026-03-16", symbol=test_symbol, international=False))
df

,earningsCalendar
0,[]


##### Covid-19

In [16]:
# Covid-19
df = pd.json_normalize(finnhub_client.covid19())
df

,state,case,death,updated
0,California,12711918,112443,2024-11-18 18:02:03
1,Texas,9190299,104793,2024-11-18 18:02:03
2,Florida,8048191,95206,2024-11-18 18:02:03
3,New York,7587861,83374,2024-11-18 18:02:03
4,Illinois,4136659,42005,2024-11-18 18:02:03
5,Georgia,3287483,44069,2024-11-18 18:02:03
6,Pennsylvania,3565499,51480,2024-11-18 18:02:03
7,Ohio,3741277,43896,2024-11-18 18:02:03
8,North Carolina,3501404,29059,2024-11-18 18:02:03
9,Arizona,2607545,34402,2024-11-18 18:02:03


##### FDA Calendar

In [18]:
# FDA Calendar
df = pd.json_normalize(finnhub_client.fda_calendar())
df

,fromDate,toDate,eventDescription,url
0,2026-04-30 08:00:00,2026-04-30 17:00:00,"April 30, 2026: Meeting of the Oncologic Drugs...",https://www.fda.gov/advisory-committees/adviso...
1,2026-03-12 09:00:00,2026-03-12 15:30:00,Vaccines and Related Biological Products Advis...,https://www.fda.gov/advisory-committees/adviso...
2,2026-01-22 09:00:00,2026-01-22 16:30:00,"January 22, 2026: Tobacco Products Scientific ...",https://www.fda.gov/advisory-committees/adviso...
3,2025-12-10 09:00:00,2025-12-10 15:30:00,UPDATED MEETING TIME AND PUBLIC PARTICIPATION ...,https://www.fda.gov/advisory-committees/adviso...
4,2025-12-03 09:00:00,2025-12-03 18:00:00,"December 3, 2025: Circulatory System Devices P...",https://www.fda.gov/advisory-committees/adviso...
5,2025-11-13 10:00:00,2025-11-13 16:00:00,Pediatric Advisory Committee Meeting Announcem...,https://www.fda.gov/advisory-committees/adviso...
6,2025-11-06 09:00:00,2025-11-06 18:00:00,"November 6, 2025: Digital Health Advisory Comm...",https://www.fda.gov/advisory-committees/adviso...
7,2025-10-09 08:30:00,2025-10-09 18:00:00,Vaccines and Related Biological Products Advis...,https://www.fda.gov/advisory-committees/adviso...
8,2025-10-08 09:00:00,2025-10-08 15:30:00,POSTPONED: General Hospital and Personal Use D...,https://www.fda.gov/advisory-committees/adviso...
9,2025-10-07 09:00:00,2025-10-07 16:30:00,"October 7, 2025: Tobacco Products Scientific A...",https://www.fda.gov/advisory-committees/adviso...


##### Symbol Lookup

In [12]:
# Symbol lookup
df = pd.json_normalize(finnhub_client.symbol_lookup(test_symbol_name))
df

,count,result
0,11,"[{'description': 'Apple Inc', 'displaySymbol':..."


In [13]:
## symbol_lookup_pre_df = df.copy(deep=True)
df.drop(columns=['count'], inplace=True)
## symbol_lookup_pre_df.rename(columns={'':'','':''}, inplace=True)
duckdb.register("df_view", df)
res = duckdb.query("SELECT \
                   CAST(json_extract(unnest(result), '$.description') AS STRING) AS description, \
                   CAST(json_extract(unnest(result), '$.displaySymbol') AS STRING) AS displaySymbol, \
                   CAST(json_extract(unnest(result), '$.symbol') AS STRING) AS symbol, \
                   CAST(json_extract(unnest(result), '$.type') AS STRING) AS type \
                   FROM df_view").to_df()  ##WHERE period == '1985-09-27'").to_df()
res.replace('"', '', regex=True, inplace=True)
res.head()

,description,displaySymbol,symbol,type
0,Apple Inc,AAPL,AAPL,Common Stock
1,Apple Hospitality REIT Inc,APLE,APLE,Common Stock
2,Apple Flavor & Fragrance Group Co Ltd,603020.SS,603020.SS,Common Stock
3,Maui Land & Pineapple Company Inc,MLP,MLP,Common Stock
4,Apple iSports Group Inc,AAPI,AAPI,Common Stock


##### Visa Application (Not Working as of Last Test)

In [14]:
# Visa application
df = pd.json_normalize(finnhub_client.stock_visa_application(test_symbol, "2015-01-01", "2022-06-15"))
df

ReadTimeout: HTTPSConnectionPool(host='api.finnhub.io', port=443): Read timed out. (read timeout=10)

##### Insider Sentiment

In [19]:
# Insider sentiment
df = pd.json_normalize(finnhub_client.stock_insider_sentiment(test_symbol, '2021-01-01', '2022-03-01'))
df

,data,symbol
0,"[{'symbol': 'AAPL', 'year': 2021, 'month': 1, ...",AAPL


In [20]:
df.drop(columns=['symbol'], inplace=True)
duckdb.register("df_view", df)
res = duckdb.query("SELECT \
                   CAST(json_extract(unnest(data), '$.symbol') AS STRING) AS symbol, \
                   CAST(json_extract(unnest(data), '$.year') AS INT) AS year, \
                   CAST(json_extract(unnest(data), '$.month') AS INT) AS month, \
                   CAST(json_extract(unnest(data), '$.change') AS FLOAT) AS change, \
                   CAST(json_extract(unnest(data), '$.mspr') AS FLOAT) AS mspr \
                   FROM df_view").to_df()  ##WHERE period == '1985-09-27'").to_df()
res.replace('"', '', regex=True, inplace=True)
res.head()

,symbol,year,month,change,mspr
0,AAPL,2021,1,281.0,100.000000
1,AAPL,2021,2,-6514.0,-8.585512
2,AAPL,2021,4,-470506.0,-27.180288
3,AAPL,2021,5,-142082.0,-100.000000
4,AAPL,2021,8,-5158896.0,-33.712353


##### USA Spending (Not Working as of Last Test)

In [22]:
# USA Spending
df = pd.json_normalize(finnhub_client.stock_usa_spending(test_symbol, "2021-01-01", "2022-06-15"))
df

ReadTimeout: HTTPSConnectionPool(host='api.finnhub.io', port=443): Read timed out. (read timeout=10)

##### Market Holiday (Not Working as of Last Test)

In [23]:
## Market Holday / Status
df = pd.json_normalize(finnhub_client.market_holiday(exchange='US'))
df


ReadTimeout: HTTPSConnectionPool(host='api.finnhub.io', port=443): Read timed out. (read timeout=10)

##### Market Status (Not Working as of Last Test)

In [24]:
df = pd.json_normalize(finnhub_client.market_status(exchange='US'))
df

ReadTimeout: HTTPSConnectionPool(host='api.finnhub.io', port=443): Read timed out. (read timeout=10)

## Paid Endpoints